# Budgerigar：双路径端到端复读

局部软写入序列保存可重放的时间细节；AutoMachine 式 token optimizer 提炼长期抽象内容。解码器同时注意两条路径。模型仍在连续时间轴中自行学会听完、保持思考间隔、再以固定人格声线复读，不使用离散行为状态。

In [ ]:
#@title 1. 更新代码
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib
if not Path(REPO_DIR).is_dir(): subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else: subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train,data]'],check=True)
sys.path.insert(0,REPO_DIR)
for name in [k for k in list(sys.modules) if k=='budgerigar' or k.startswith('budgerigar.')]: del sys.modules[name]
importlib.invalidate_caches()
commit=subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('commit:',commit)

In [ ]:
#@title 2. Drive 与已有内容 checkpoint
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar')
FEATURE_FINGERPRINT='f1f2ace085a17835' #@param {type:'string'}
FEATURE_MANIFEST=WORK_ROOT/'manifests'/f'cmu_arctic.features.{FEATURE_FINGERPRINT}.jsonl'
STATS_PATH=WORK_ROOT/'features'/f'stats.smoke64.arctic_slt.{FEATURE_FINGERPRINT}.pt'
CONTENT_CHECKPOINT=WORK_ROOT/'checkpoints'/f'content_local_dual_ctc_{FEATURE_FINGERPRINT}'/'best.pt'
for path in (FEATURE_MANIFEST,STATS_PATH,CONTENT_CHECKPOINT): assert path.is_file(),path
import torch,json
stats=torch.load(STATS_PATH,map_location='cpu',weights_only=True)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
#@title 3. 双路径结构检查
from budgerigar.echo_data import load_pairs,EchoEpisodeDataset,collate_episodes
from budgerigar.dual_path_echo import DualPathEchoConfig,create_dual_path_echo
config=DualPathEchoConfig(hidden_dim=192,local_slots=160,abstract_slots=160,update_stride=4,local_encoder_layers=4)
model=create_dual_path_echo(config)
print(f'parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')
pairs=[p for p in load_pairs(FEATURE_MANIFEST,'arctic_slt') if p.split=='train']
preview=EchoEpisodeDataset(pairs[:1],stats,(16,28),preload=True)
inputs,targets,voice,metadata=preview[0]
with torch.no_grad(): result=model(inputs[:64].unsqueeze(0))
print(inputs.shape,targets.shape,result[0].shape,result[3]['local_write'].shape)

In [ ]:
#@title 4. T4 端到端 smoke training
MAX_STEPS=200 #@param {type:'integer'}
BATCH_SIZE=2 #@param {type:'integer'}
if not torch.cuda.is_available(): raise RuntimeError('请选择 GPU runtime')
from budgerigar.train_dual_path import DualPathTrainingConfig,train_dual_path_echo
RUN_DIR=WORK_ROOT/'checkpoints'/f'dual_path_echo_arctic_slt_{FEATURE_FINGERPRINT}'
training=DualPathTrainingConfig(batch_size=BATCH_SIZE,max_steps=MAX_STEPS,initialization_checkpoint=str(CONTENT_CHECKPOINT))
report=train_dual_path_echo(FEATURE_MANIFEST,RUN_DIR,training,config,stats)
print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 5. 时间轴、真实内容与路径消融联合评估
from budgerigar.evaluate_echo import evaluate_checkpoint
from budgerigar.evaluate_content import evaluate_content
CHECKPOINT=RUN_DIR/'best.pt'
EVAL_DIR=RUN_DIR/'joint_evaluation'
behavior=evaluate_checkpoint(CHECKPOINT,FEATURE_MANIFEST,EVAL_DIR,max_pairs=32)
content=evaluate_content(CHECKPOINT,FEATURE_MANIFEST,EVAL_DIR,max_pairs=32,candidates=16)
best=min(report['history'],key=lambda row:row['validation_repeat_l1'])
ablation_pass=best['no_local_degradation']>0.01 and best['no_abstract_degradation']>0.005
combined_pass=behavior['behavior_pass'] and content['content_pass'] and ablation_pass
print(json.dumps({'behavior_pass':behavior['behavior_pass'],'content_pass':content['content_pass'],'ablation_pass':ablation_pass,'combined_pass':combined_pass,'best_ablation':best},ensure_ascii=False,indent=2))
if not combined_pass: print('未通过：不扩大训练，按端到端失败指标调整。')